In [30]:
import sympy as sym
from microscope_calibration.common.model import Parameters4DSTEM, Model4DSTEM, DescanError, PixelYX, symbol_maker, trace, is_sympy

In [2]:
params = symbol_maker(Parameters4DSTEM, postfix=None, recurse_for=[DescanError, PixelYX])

In [3]:
params

Parameters4DSTEM(overfocus=overfocus_0, scan_pixel_pitch=scan_pixel_pitch_1, scan_center=PixelYX(y=y_2, x=x_3), scan_rotation=scan_rotation_4, camera_length=camera_length_5, detector_pixel_pitch=detector_pixel_pitch_6, detector_center=PixelYX(y=y_7, x=x_8), semiconv=semiconv_9, flip_factor=flip_factor_10, descan_error=DescanError(pxo_pxi=pxo_pxi_11, pxo_pyi=pxo_pyi_12, pyo_pxi=pyo_pxi_13, pyo_pyi=pyo_pyi_14, sxo_pxi=sxo_pxi_15, sxo_pyi=sxo_pyi_16, syo_pxi=syo_pxi_17, syo_pyi=syo_pyi_18, offpxi=offpxi_19, offpyi=offpyi_20, offsxi=offsxi_21, offsyi=offsyi_22), detector_rotation=detector_rotation_23)

In [5]:
scan_pos = symbol_maker(PixelYX, postfix=None)
source_dy, source_dx = sym.symbols('source_{dy} source_{dx}')

In [6]:
res = trace(
    params=params,
    scan_pos=scan_pos,
    source_dy=source_dy,
    source_dx=source_dx,
)

AttributeError: 'MutableDenseMatrix' object has no attribute 'to_matrix'

In [13]:
res

OrderedDict([('source',
              ResultSection(component=PointSource(z=0, semi_conv=semiconv, offset_xy=CoordsXY(x=0.0, y=0.0)), ray=Ray(x=0, y=0, dx=source_{dx}, dy=source_{dy}, z=0.0, pathlength=0.0, _one=1.0), sampling=None)),
             ('overfocus',
              ResultSection(component=Propagator(distance=overfocus, propagator=<temgym_core.propagator.FreeSpaceParaxial object at 0x7f62c6d99940>), ray=Ray(x=overfocus*source_{dx}, y=overfocus*source_{dy}, dx=source_{dx}, dy=source_{dy}, z=overfocus, pathlength=overfocus, _one=1.0), sampling=None)),
             ('scanner',
              ResultSection(component=Scanner(z=overfocus, scan_pos_x=0, scan_pos_y=0, scan_tilt_x=0.0, scan_tilt_y=0.0), ray=Ray(x=overfocus*source_{dx}, y=overfocus*source_{dy}, dx=source_{dx}, dy=source_{dy}, z=overfocus, pathlength=overfocus, _one=1.0), sampling=None)),
             ('specimen',
              ResultSection(component=Plane(z=overfocus), ray=Ray(x=overfocus*source_{dx}, y=overfocus*source

In [14]:
f = sym.lambdify((params, scan_pos, source_dy, source_dx), res)

TypeError: 'Zero' object is not subscriptable

In [12]:
import jax_dataclasses as jdc

In [13]:
@jdc.pytree_dataclass
class Ray:
    x: float
    y: float

In [14]:
@jdc.pytree_dataclass
class ZRay(Ray):
    z: float

In [16]:
ZRay(x=0., y=0., z=0.)

ZRay(x=0.0, y=0.0, z=0.0)

In [103]:
def mytrace(r: ZRay):
    return ZRay(
        x=r.x + 1,
        y=r.y + 1,
        z="abc",
    )

In [104]:
r = ZRay(*sym.symbols('x, y'), 1)

In [105]:
r

ZRay(x=x, y=y, z=1)

In [106]:
res = mytrace(r)

In [107]:
res

ZRay(x=x + 1, y=y + 1, z='abc')

In [108]:
mytrace(ZRay(1, 2, 3))

ZRay(x=2, y=3, z='abc')

In [109]:
import jax.tree

In [110]:
from copy import deepcopy
from typing import TypeVar, Callable

In [111]:
SymbolJaxTree = TypeVar("SymbolJaxTree")


def lambdify(inp: SymbolJaxTree, func: Callable[[SymbolJaxTree], SymbolJaxTree], **kwargs):
    outp = func(inp)
    inp_leaves, inp_treedef = jax.tree.flatten(inp)
    outp_leaves, outp_treedef = jax.tree.flatten(outp)

    inp_indices = []
    inp_symbols = []
    inp_dups = {}

    for i, leave in enumerate(inp_leaves):
        if isinstance(leave, sym.Symbol):
            if leave in inp_symbols:
                inp_dups[i] = inp_symbols.index(leave)
            else:
                inp_indices.append(i)
                inp_symbols.append(leave)
        elif is_sympy(leave) and not isinstance(leave, (sym.NumberSymbol, sym.Number)):
            raise ValueError(
                f"SymPy leave {leave} found that is not a symbol or a constant number. "
                "Only symbols and constants are allowed in the input definition."
            )

    outp_indices = []
    outp_exprs = []

    for i, leave in enumerate(outp_leaves):
        if isinstance(leave, sym.Basic):
            outp_indices.append(i)
            outp_exprs.append(leave)

    inp_indices_set = set(inp_indices)
    inner_f = sym.lambdify(inp_symbols, outp_exprs, **kwargs)

    def outer(ii):
        ii_leaves, ii_treedef = jax.tree.flatten_with_path(ii)
        if ii_treedef != inp_treedef:
            raise ValueError(
                f'Tree definition of input {ii_treedef} does not match expected '
                f'tree definition {inp_treedef}.'
            )
        for i, (path, leave) in enumerate(ii_leaves):
            if i not in inp_indices_set:
                if i in inp_dups:
                    orig_i = inp_dups[i]
                    orig_path, orig_leave = ii_leaves[orig_i]
                    if orig_leave != leave:
                        raise ValueError(
                            f"Input value {leave} with path {path} was a duplicate symbol in original input "
                            f"but is now not matching the input value {orig_leave} at {orig_path}"
                        )
                elif leave != inp_leaves[i]:
                    raise ValueError(
                        f"Constant value {leave} doesn't match reference input "
                        f"{inp_leaves[i]} for {path}.")

        ii_vals = [ii_leaves[i][1] for i in inp_indices]
        oo_inner = inner_f(*ii_vals)
        outp = deepcopy(outp_leaves)
        for i, val in enumerate(oo_inner):
            index = outp_indices[i]
            outp[index] = val
        return jax.tree.unflatten(outp_treedef, outp)

    return outer

In [112]:
x, y, z = sym.symbols('x y z')

In [113]:
f = lambdify(ZRay(x, y, z), mytrace, cse=True)

In [114]:
f(ZRay(x=4, y=3, z=4))

ZRay(x=5, y=4, z='abc')

In [118]:
type(sym.sympify("abc"))

sympy.core.symbol.Symbol

In [116]:
expr = (x + y)**2 + z

In [117]:
expr.atoms()

{2, x, y, z}

In [150]:
expr.free_symbols

{x, y, z}

In [120]:
m = sym.ImmutableMatrix(
    [
        (x, 0),
        (0, y),
    ]
)

In [122]:
m[0, 0]

x